Re-analysis of existing dumps. **Accelerator: None** — this costs no GPU quota.

Add Input → Your Work → Notebook Output → the run notebook, so its `results/` is attached.

In [1]:
REPO_URL = "https://github.com/Splestule/candidate_reranker.git"
BRANCH = "main"

In [2]:
import subprocess, sys
from pathlib import Path

CODE = Path("/kaggle/working/candidate_reranker")
if not (CODE / ".git").exists():
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(CODE)],
                   check=True)

sys.path.insert(0, str(CODE / "src"))
import kaggle_env as K

COMMIT = K.sync(REPO_URL, BRANCH)     # rerun this cell after every push
env = K.prepare_cpu(COMMIT)
DUMPS = K.find_dumps()

Cloning into '/kaggle/working/candidate_reranker'...
From https://github.com/Splestule/candidate_reranker
 * branch            main       -> FETCH_HEAD


HEAD is now at 97ce213 allocate candidate budget by utterance length
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 38.4 MB/s eta 0:00:00
commit   97ce213
test-clean-k30 /kaggle/input/notebooks/eduardimon/k30-test/results/test-clean-k30-b36958a.jsonl
dev-clean    /kaggle/input/notebooks/eduardimon/rover-word-reranker-test-dialects/results/dev-clean-7262276.jsonl
dialects     /kaggle/input/notebooks/eduardimon/rover-word-reranker-test-dialects/results/dialects-7262276.jsonl
test-clean   /kaggle/input/notebooks/eduardimon/rover-word-reranker-test-dialects/results/test-clean-7262276.jsonl
test-other   /kaggle/input/notebooks/eduardimon/rover-word-reranker-test-dialects/results/test-other-7262276.jsonl


Tune on dev-clean. `gamma` is the new knob: 1.0 is one vote per candidate, 0.0 is one vote per cluster of near-identical candidates.

In [3]:
K.run(env, "tune.py",
      "--dev", DUMPS["dev-clean"],
      "--test", DUMPS["test-clean"],
      "--json", env.results / f"tune-{COMMIT}.json")

dev: 800 utterances from 1 dump(s)

lambda for conf + lambda*mbr
   0.0    7.93
   0.1    7.55
  0.25    7.29
   0.5    7.20
  0.75    7.11
   1.0    7.09
   1.5    7.08
   2.0    7.20
   3.0    7.22

ROVER: alpha (1.0 = votes only), epsilon confidence, gamma (1.0 = one vote each)
  alpha 0.5   eps 0.7   gamma 0.5     5.68
  alpha 0.5   eps 0.7   gamma 0.75    5.68
  alpha 0.5   eps 0.7   gamma 1.0     5.69
  alpha 0.5   eps 0.7   gamma 0.25    5.70
  alpha 0.3   eps 0.7   gamma 0.75    5.75
  alpha 0.3   eps 0.7   gamma 1.0     5.75
  alpha 0.5   eps 0.5   gamma 0.75    5.76
  alpha 0.5   eps 0.5   gamma 1.0     5.77
  alpha 0.5   eps 0.5   gamma 0.5     5.77
  alpha 0.7   eps 0.7   gamma 1.0     5.78
  alpha 0.7   eps 0.7   gamma 0.75    5.80
  alpha 0.7   eps 0.5   gamma 1.0     5.80
  alpha 0.7   eps 0.5   gamma 0.75    5.80
  alpha 0.85  eps 0.7   gamma 1.0     5.80
  alpha 0.7   eps 0.5   gamma 0.5     5.82
  alpha 0.5   eps 0.7   gamma 0.0     5.83
  alpha 0.85  eps 0.7   gamma 

0

Measure on all three test sets with the tuned parameters, with paired bootstrap intervals.

In [4]:
import json

tuned = json.load(open(env.results / f"tune-{COMMIT}.json"))
print({k: tuned[k] for k in ("lambda", "alpha", "eps_conf", "gamma")})

for tag in ["test-clean", "test-other", "dialects"]:
    print("\n" + "#" * 74 + f"\n# {tag}\n" + "#" * 74)
    K.run(env, "analyze_compose.py", DUMPS[tag],
          "--alpha", tuned["alpha"], "--eps_conf", tuned["eps_conf"],
          "--gamma", tuned["gamma"],
          "--json", env.results / f"compose-{tag}-{COMMIT}.json")

{'lambda': 1.5, 'alpha': 0.5, 'eps_conf': 0.7, 'gamma': 0.5}

##########################################################################
# test-clean
##########################################################################
COMPOSITION  ·  /kaggle/input/notebooks/eduardimon/rover-word-reranker-test-dialects/results/test-clean-7262276.jsonl
utterances: 2620   per-word confidences: yes
ROVER alpha 0.5  eps_conf 0.7  gamma 0.5

                                    corpus WER   mean-utt
pick by confidence                        9.04       8.24
pick by conf + 0.5*mbr                    8.36       7.65
ROVER, word-level vote                    6.47       6.64
oracle over whole candidates              6.75       5.80
oracle over word combinations             3.25       3.40
  same, unrelated alternatives            6.36       5.45

combining beats the best single candidate on 34.5 % of utterances
headroom beyond the candidate oracle: 3.50 points
same for the control: 0.39 points
signal-to-luc

What the cluster weighting is worth on its own: the same run at `gamma = 1.0`, everything else identical.

In [5]:
for tag in ["test-clean", "test-other", "dialects"]:
    print("\n" + "#" * 74 + f"\n# {tag}  ·  gamma 1.0\n" + "#" * 74)
    K.run(env, "analyze_compose.py", DUMPS[tag],
          "--alpha", tuned["alpha"], "--eps_conf", tuned["eps_conf"], "--gamma", 1.0,
          "--n_boot", 1000,
          "--json", env.results / f"compose-{tag}-gamma1-{COMMIT}.json")


##########################################################################
# test-clean  ·  gamma 1.0
##########################################################################
COMPOSITION  ·  /kaggle/input/notebooks/eduardimon/rover-word-reranker-test-dialects/results/test-clean-7262276.jsonl
utterances: 2620   per-word confidences: yes
ROVER alpha 0.5  eps_conf 0.7  gamma 1.0

                                    corpus WER   mean-utt
pick by confidence                        9.04       8.24
pick by conf + 0.5*mbr                    8.36       7.65
ROVER, word-level vote                    6.53       6.73
oracle over whole candidates              6.75       5.80
oracle over word combinations             3.25       3.40
  same, unrelated alternatives            6.36       5.45

combining beats the best single candidate on 34.5 % of utterances
headroom beyond the candidate oracle: 3.50 points
same for the control: 0.39 points
signal-to-luck ratio: 9.1x
ROVER vs baseline: +2.51 points



Spend the same candidate budget unevenly. Needs the K = 30 dump attached as well, since a policy can only hand out candidates the dump actually holds.

In [6]:
BIG = DUMPS.get("test-clean-k30")
if BIG is None:
    print("attach the k30 notebook output too, then rerun this cell")
else:
    K_env = K   # same module, clearer name here
    K.run(env, "allocate.py", BIG,
          "--budget", 15, "--k_min", 2, "--k_max", 30,
          "--alpha", tuned["alpha"], "--eps_conf", tuned["eps_conf"],
          "--json", env.results / f"allocate-{COMMIT}.json")

2620 utterances, dump holds K = 30, budget 15.0 mean candidates, k in [2, 30]

policy            corpus WER   mean k     k range
const                   6.53    15.00       15-15
duration                6.41    15.00        3-30
words                   6.41    15.00        2-30
unconf                  6.49    15.00        2-30
words_unconf            6.46    15.00        2-30

--------------------------------------------------------------
PAIRED BOOTSTRAP against the flat budget
--------------------------------------------------------------
duration over const                 +0.12  95% CI [+0.03, +0.23]   better 163 / worse 139 / tied 2318
words over const                    +0.12  95% CI [+0.02, +0.22]   better 164 / worse 143 / tied 2313
unconf over const                   +0.04  95% CI [-0.06, +0.14]   better 165 / worse 174 / tied 2281
                                  interval spans zero: not distinguishable
words_unconf over const             +0.07  95% CI [-0.03, +0.19]   bette

Copy the small result files into the repo checkout so they can be committed. Download them from the notebook output, or push from a machine with credentials.

In [7]:
import shutil

keep = CODE / "results"
keep.mkdir(exist_ok=True)
for p in sorted(env.results.glob("*.json")):
    shutil.copy(p, keep / p.name)
    print(keep / p.name)

/kaggle/working/candidate_reranker/results/allocate-97ce213.json
/kaggle/working/candidate_reranker/results/compose-dialects-97ce213.json
/kaggle/working/candidate_reranker/results/compose-dialects-gamma1-97ce213.json
/kaggle/working/candidate_reranker/results/compose-test-clean-97ce213.json
/kaggle/working/candidate_reranker/results/compose-test-clean-gamma1-97ce213.json
/kaggle/working/candidate_reranker/results/compose-test-other-97ce213.json
/kaggle/working/candidate_reranker/results/compose-test-other-gamma1-97ce213.json
/kaggle/working/candidate_reranker/results/tune-97ce213.json
